In [19]:
# ================================================
# initialize
# ================================================
include("setup_notebook.jl")


MECH3620 Loading modules...
[OK] PyCall: C:/Users/kychandv/miniconda3/envs/mech3620/python.exe
TA_code path: c:\Users\kychandv\MECH3620\3620_project\MECH3620_Project\TA_code
TA_code exists: true

Importing functions...
[OK] Imported mech3620_models module
Available in mech3620_models:
[ERROR] Cannot import mech3620_models: MethodError(names, (PyObject <module 'mech3620_models' from 'c:\\Users\\kychandv\\MECH3620\\3620_project\\MECH3620_Project\\TA_code\\mech3620_models.py'>,), 0x000000000000981f)
[OK] calc_thrust_lapse loaded
[OK] US_Standard_1976_Atmosphere loaded
[OK] calculate_time_to_climb loaded
[OK] calc_TOP_given_BFL_requirement loaded
[OK] Atmosphere instance created

Initialization complete!

Available functions/variables:
  ✓ calc_thrust_lapse
  ✓ US_Standard_1976_Atmosphere (class)
  ✓ atmosphere (instance)
  ✓ calc_TOP


"""
Only direct operating cost (DOC) need to cover for project

DOC = COC + FOC
"""

In [20]:
#==========================#
# Cash operating costs
#==========================#

function base_then_CDF(b_year, t_year)
    b_CEF = 5.17053+0.104981(b_year-2006)   # base CEF
    t_CEF = 5.17053+0.104981(t_year-2006)   # then CEF
    CEF = t_CEF/b_CEF   #effective CEF
    return b_CEF, t_CEF, CEF
end

base_then_CDF (generic function with 1 method)

In [21]:
#=========================#
# 1. Crew costs
#=========================#

function crew_costs(MTOW, t_b, CEF) #$/Flight 
    crew_cost = (482 + 0.590(MTOW/1000))*t_b * CEF
    return crew_cost
end

crew_costs (generic function with 1 method)

In [22]:
#=========================#
# 2. Attendants costs
#=========================#

function attendants_costs(n_attd, t_b, CEF) #$/Flight 
    attendants_cost = (78 * n_attd) * t_b * CEF
    return attendants_cost
end

attendants_costs (generic function with 1 method)

In [23]:
#=========================#
# 3. Fuel Cost
#=========================#

function fuel_costs(W_f, rho_f, P_f) #$/Flight 
    fuel_cost = 1.02 * W_f * (P_f / rho_f)
    return fuel_cost
end

fuel_costs (generic function with 1 method)

In [24]:
#=========================#
# 4. Oil cost
#=========================#

function oil_costs(rho_o, P_o, t_b, W_f)  # $/Flight
    W_o = 0.0125 * W_f * (t_b/100)
    oil_cost = 1.02 * W_o * (P_o / rho_o)
    return oil_cost
end

oil_costs (generic function with 1 method)

In [25]:
#=========================#
# 5. Landing fees
#=========================#

function landing_fees(MTOW, CEF)    # $/Flight
    landing_fee = 4.25 * (MTOW / 1000) * CEF
    return landing_fee
end

landing_fees (generic function with 1 method)

In [26]:
#=========================#
# 6. Navigation fees
#=========================#

function navigation_fees(MTOW, CEF)    # $/Flight
    navigation_fee = 68 * sqrt(MTOW/1000) * CEF
    return navigation_fee
end

navigation_fees (generic function with 1 method)

In [27]:
#=========================#
# 7. Airframe maintenance costs
#=========================#

function airframe_maintenance_costs(MTOW, t_b, CEF, R_L)   # $/Flight
    P_aircraft = 10^(3.3191+0.8043*log10(MTOW))*CEF
    P_engine = 10^(2.3044+0.8858*log10(MTOW))*CEF
    P_airframe = P_aircraft - P_engine
    C_ML = 1.03 * (3 + 0.067 * MTOW/1000) * R_L
    C_MM = 1.03 * 30 * CEF + (0.79e-5) * P_airframe
    airframe_maintenance_cost = (C_ML + C_MM)* t_b
    return airframe_maintenance_cost, P_aircraft, P_engine, P_airframe
end

airframe_maintenance_costs (generic function with 1 method)

In [28]:
#=============================#
# 8. Engine maintenance costs
#=============================#
global n_eng = 2

function engine_maintenance_costs(T_o, t_b, R_L)   # $/Flight
    C_ML = (0.645 + 0.05 * T_o/ 10^4)*(0.566 + 0.434/t_b) * R_L
    C_MM = (25+ 18*T_o/10^4) * (0.62 + 038/t_b) * CEF
    engine_maintenance_cost = n_eng * (C_ML + C_MM)* t_b
    return engine_maintenance_cost
end

engine_maintenance_costs (generic function with 1 method)

In [29]:
#============================#
# 9. Insurance cost
#============================#
global IR_a = 0.02 # 2%
function insurance_costs(t_b, P_aircraft)   # $/Flight
    U_annual = 1.5e3 * (3.4546 * t_b + 2.994 - (12.289 *t_b^2 - 5.6626*t_b + 8.964)^0.5)#in hr
    insurance_cost = (IR_a * P_aircraft / U_annual) * t_b
    return insurance_cost, U_annual
end

insurance_costs (generic function with 1 method)

In [30]:
#============================#
# 10. Financing cost
#============================#
global R_f = 0.05
function financing_costs(t_b, P_aircraft, U_annual, R_f)   # $/Flight
financing_cost = (R_f * P_aircraft / U_annual) * t_b
return financing_cost
end 

financing_costs (generic function with 1 method)

In [31]:
#=============================#
# 11. Depreciation cost
#=============================#
global K_depreciation = 0.3    # From note
global n = 20  # years of operation form note
 function Depreciation_costs(P_aircraft, K_depreciation, t_b, U_annual)  #$/Flight
    depreciation_cost = ((1-K_depreciation) * P_aircraft / (n*U_annual)) * t_b
    return depreciation_cost
 end


Depreciation_costs (generic function with 1 method)

In [32]:
#=========================#
# 12. Registration fees
#=========================#

function registration_fees(MTOW, CEF, DOC)    # $/Flight
    registration_fee = (0.001+(10^(-8))*MTOW) * DOC
    return registration_fee
end

registration_fees (generic function with 1 method)

In [40]:
b_year = 1993
t_year = 2026
kg_to_lb = 2.20462 # lbs/kg
#----------------------------------------------------------------
#unit transformation
#----------------------------------------------------------------
kgm3_to_lbgal = 0.00835 # lbs/gal
bbl_to_gal = 42 # gal/bbl
#-----------------------------------------------------------
CEF = base_then_CDF(b_year, t_year)[3]
b_CEF = base_then_CDF(b_year, t_year)[1]
t_CEF = base_then_CDF(b_year, t_year)[2]

MTOW = 33614.1*kg_to_lb #lbs, from 02
#1.--------------------------------------------------------------
t_b = 2.5      # flight block time, in hrs
crew_cost = crew_costs(MTOW, t_b, CEF)
#2.--------------------------------------------------------------
n_attd = 2      # no of flight attendants
attendants_cost = attendants_costs(n_attd, t_b, CEF)
#3.-------------------------------------------------------------
W_f = 11183.3*kg_to_lb #lbs, from 02, fuel weight
rho_f = 807.5 * kgm3_to_lbgal    # fuel density, lbs/gal, 775-840 g/L,take average here
P_f = 196.73/bbl_to_gal      # price of fuel USD/gal
fuel_cost = fuel_costs(W_f, rho_f, P_f)
#4.-------------------------------------------------------------
rho_o =  995* kgm3_to_lbgal  # oil density, lbs/gal, Eastman Turbo oil 25
P_o = 108.64      # price of oil USD/gal
oil_cost = oil_costs(rho_o, P_o, t_b, W_f)
#5.-------------------------------------------------------------
landing_fee = landing_fees(MTOW, CEF)
#6.-------------------------------------------------------------
navigation_fee = navigation_fees(MTOW, CEF)
#7.-------------------------------------------------------------
R_L = 28.72 # maintenance labor rate in USD/hr for the year of interest, Take the aircraft mechanics as reference
airframe_maintenance_cost, P_aircraft, P_engine, P_airframe = airframe_maintenance_costs(MTOW, t_b, CEF, R_L)
#8.-------------------------------------------------------------
T_o = 14500 # max engine thrust in lbs, CF 34-8E under wing
engine_maintenance_cost = engine_maintenance_costs(T_o, t_b, R_L)
#9.-------------------------------------------------------------
insurance_cost_per_flight, U_annual = insurance_costs(t_b, P_aircraft)
#10.-------------------------------------------------------------
financing_cost = financing_costs(t_b, P_aircraft, U_annual, R_f)
#11.-------------------------------------------------------------
depreciation_cost = Depreciation_costs(P_aircraft, K_depreciation, t_b, U_annual)


# Sum of all 1-11 cost-------------------------------------------------------------
DOC = crew_cost + attendants_cost + fuel_cost + oil_cost + landing_fee + navigation_fee + airframe_maintenance_cost + engine_maintenance_cost + insurance_cost_per_flight + financing_cost + depreciation_cost


#12.-------------------------------------------------------------
registration_fee = registration_fees(MTOW, CEF, DOC)

# Total cost
total_cost = DOC + registration_fee

#Print results
println("Crew cost: ", crew_cost, " Unit: \$/Flight")
println("Attendants cost: ", attendants_cost, " Unit: \$/Flight")
println("Fuel cost: ", fuel_cost, " Unit: \$/Flight")
println("Oil cost: ", oil_cost, " Unit: \$/Flight")
println("Landing fee: ", landing_fee, " Unit: \$/Flight")
println("Navigation fee: ", navigation_fee, " Unit: \$/Flight")
println("Airframe maintenance cost: ", airframe_maintenance_cost, " Unit: \$/Flight")
println("Engine maintenance cost: ", engine_maintenance_cost, " Unit: \$/Flight")
println("Insurance cost: ", insurance_cost_per_flight, " Unit: \$/Flight")
println("Financing cost: ", financing_cost, " Unit: \$/Flight")
println("Depreciation cost: ", depreciation_cost, " Unit: \$/Flight")
println("Direct operating cost (DOC): ", DOC/2.5, " Unit: \$/hr")
println("Registration fee: ", registration_fee, " Unit: \$/Flight")
println("Total cost: ", total_cost, " Unit: \$/Flight")
println("Total cost per year: ", total_cost * 3250, " Unit: \$/Year")

Crew cost: 2510.711403036915 Unit: $/Flight
Attendants cost: 745.0143558069747 Unit: $/Flight
Fuel cost: 17470.13194692208 Unit: $/Flight
Oil cost: 102.76237014414998 Unit: $/Flight
Landing fee: 601.650379586645 Unit: $/Flight
Navigation fee: 1118.2439632516523 Unit: $/Flight
Airframe maintenance cost: 1229.5441051439802 Unit: $/Flight
Engine maintenance cost: 7797.627505044485 Unit: $/Flight
Insurance cost: 346.0325101099574 Unit: $/Flight
Financing cost: 865.0812752748935 Unit: $/Flight
Depreciation cost: 605.5568926924254 Unit: $/Flight
Direct operating cost (DOC): 13356.942682805666 Unit: $/hr
Registration fee: 58.138202469501984 Unit: $/Flight
Total cost: 33450.49490948367 Unit: $/Flight
Total cost per year: 1.0871410845582193e8 Unit: $/Year
